In [1]:
import os
import openai
from dotenv import load_dotenv, find_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')

In [2]:
from ragas.llms import LangchainLLMWrapper
from langchain_openai import ChatOpenAI
from ragas.embeddings import OpenAIEmbeddings
import openai

generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o"))
openai_client = openai.OpenAI()
generator_embeddings = OpenAIEmbeddings(client=openai_client)

c:\Users\abhim\OneDrive\Documents\Projects\RAG evaluation with RAGAS\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\abhim\AppData\Local\Temp\ipykernel_15500\300379323.py:6: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o"))


In [28]:
from langchain_community.document_loaders import PyPDFLoader
file_path = r"C:\Users\abhim\OneDrive\Documents\Projects\RAG evaluation with RAGAS\Business Statistics - A. Aczel, J. Sounderpandian.pdf"
loader = PyPDFLoader(file_path)
pages = loader.load()

In [29]:
docs = pages[48:100]

In [30]:
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter

ragas_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,      # tokens-ish
    chunk_overlap=100
)

docs_for_ragas = ragas_splitter.split_documents(docs)


In [ ]:
from langchain_community.docstore.document import Document

# Assuming 'docs' is your list of Document objects provided in the context
all_langchain_docs = []

for doc in docs:
    
    cleaned_content = doc.page_content
    metadata = doc.metadata
    
    # Create the new LangChain Document
    langchain_doc = Document(page_content=cleaned_content, metadata=metadata)
    
    # Store it in your list
    all_langchain_docs.append(langchain_doc)

print(f"Successfully converted {len(all_langchain_docs)} documents.")
print("Example of first doc:", all_langchain_docs[0])

Successfully converted 52 documents.
Example of first doc: page_content='Aczel−Sounderpandian: 
Complete Business 
Statistics, Seventh Edition
1. Introduction and 
Descriptive Statistics
Text
45© The McGraw−Hill 
Companies, 2009
1–65. Find the 90th percentile, the quartiles, and the range of the data in problem
1–63.
1–66. The following data are numbers of color television sets manufactured per
day at a given plant: 15, 16, 18, 19, 14, 12, 22, 23, 25, 20, 32, 1 7, 34, 25, 40, 41. Draw
a frequency polygon and an ogive for these data.
1–67 . Construct a stem-and-leaf display for the data in problem 1–66.
1–68. Construct a box plot for the data in problem 1–66. What can you say about
the data?
1–69. The following data are the number of cars passing a point on a highway per
minute: 10, 12, 11, 19, 22, 21, 23, 22, 24, 25, 23, 21, 28, 26, 27, 27, 29, 26, 22, 28, 30,
32, 25, 37, 34, 35, 62. Construct a stem-and-leaf display of these data. What does the
display tell you about the data?
1–70. F

In [ ]:
from ragas.testset import TestsetGenerator
from ragas.testset.transforms import default_transforms

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs_for_ragas,testset_size=5,with_debugging_logs=True)

Applying CustomNodeFilter:   0%|          | 1/223 [00:02<09:16,  2.51s/it]Node ef4805ff-b0a0-4862-8821-247e90821314 does not have a summary. Skipping filtering.
Node e780477f-e2f1-40ff-bf51-936fbe014e22 does not have a summary. Skipping filtering.
Node a766b415-b50e-4e98-8885-c757e5c4d8ef does not have a summary. Skipping filtering.
Node b9ecbef6-7160-4800-81f9-1f32d89c98fd does not have a summary. Skipping filtering.
Applying EmbeddingExtractor:   0%|          | 0/209 [00:00<?, ?it/s]c:\Users\abhim\OneDrive\Documents\Projects\RAG evaluation with RAGAS\.venv\Lib\site-packages\ragas\testset\transforms\base.py:198: UserWarning: Using sync embedding model OpenAIEmbeddings in async context. This may impact performance. Consider using an async-compatible embedding model for better performance.
  property_name, property_value = await self.extract(node)
Generating Samples: 100%|██████████| 6/6 [00:10<00:00,  1.83s/it]


In [35]:
dataset.to_pandas()

,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name
0,What Seventh Edition about?,[Aczel−Sounderpandian: \nComplete Business \nS...,The Seventh Edition refers to 'Complete Busine...,Financial Data Analyst,POOR_GRAMMAR,SHORT,single_hop_specific_query_synthesizer
1,What was the stock price percentage for Lehman...,[1–72. The following are a sample of Motorola’...,The stock price percentage for Lehman Brothers...,Financial Data Analyst,PERFECT_GRAMMAR,MEDIUM,single_hop_specific_query_synthesizer
2,How do business statistics relate to calculati...,[<1-hop>\n\nAczel−Sounderpandian: \nComplete B...,Business statistics involve the use of mathema...,NaN,NaN,NaN,multi_hop_abstract_query_synthesizer
3,How does Bayes' Theorem explain the low probab...,[<1-hop>\n\nAczel−Sounderpandian: \nComplete B...,Bayes' Theorem explains the low probability of...,NaN,NaN,NaN,multi_hop_abstract_query_synthesizer
4,How does Bayes’ Theorem facilitate the reversa...,[<1-hop>\n\nAczel−Sounderpandian: \nComplete B...,Bayes’ Theorem allows for the reversal of cond...,NaN,NaN,NaN,multi_hop_specific_query_synthesizer
5,How does the volatility of the NASDAQ index in...,[<1-hop>\n\nlooking at the plot. Report the st...,To compare the volatility of the NASDAQ index ...,NaN,NaN,NaN,multi_hop_specific_query_synthesizer


In [39]:
import json

DOCS_CACHE_FILE = r"C:\Users\abhim\OneDrive\Documents\Projects\RAG evaluation with RAGAS\all_collected_docs_cache_new.json"

# Cache the processed documents
serializable_docs = [
    {"page_content": doc.page_content, "metadata": doc.metadata}
    for doc in docs_for_ragas
]
with open(DOCS_CACHE_FILE, 'w', encoding='utf-8') as f:
    json.dump(serializable_docs, f, ensure_ascii=False, indent=2)
print(f"Documents cached to {DOCS_CACHE_FILE}")

Documents cached to C:\Users\abhim\OneDrive\Documents\Projects\RAG evaluation with RAGAS\all_collected_docs_cache_new.json


In [8]:
from ragas.testset.graph import KnowledgeGraph

kg = KnowledgeGraph()

In [10]:
from ragas.testset.graph import Node, NodeType

for doc in docs:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
        )
    )

In [ ]:
from ragas.testset.transforms import default_transforms, apply_transforms


# define your LLM and Embedding Model
# here we are using the same LLM and Embedding Model that we used to generate the testset
transformer_llm = generator_llm
embedding_model = generator_embeddings

trans = default_transforms(documents=docs, llm=transformer_llm, embedding_model=embedding_model)
apply_transforms(kg, trans)

TypeError: apply_transforms() got an unexpected keyword argument 'with_debugging_logs'

In [ ]:
kg.save("knowledge_graph.json")
loaded_kg = KnowledgeGraph.load("knowledge_graph.json")
loaded_kg